In [ ]:
import sympy as sp
from sympy import diff, sin, cos, exp, sqrt, Rational

In [ ]:
xij, yij, dxij, dyij, rij, θij = sp.symbols('x_{ij} y_{ij} δx_{ij}, δy_{ij}, r_{ij} θ_{ij}')
me, e, t, eps0, pi = sp.symbols('m_e e t ε_0 π')

In this notebook we'll use `sympy` to do some of the heavy lifting in determining the equations of motion. The equations of motion are derived in https://journals.aps.org/prx/supplemental/10.1103/PhysRevX.6.011031/supplement.pdf , but some of the math-heavy steps are missing. Those steps are shown in the following two sections.

# Linearization of $F_x$ for the EOM in $\delta x_i$

A crucial part of the equations of motion is the electron-electron interaction term. For the perturbation in the coordinate $x_i$ we need to linearize the electron-electron interaction force in the x-direction. This force is given by `F_x` and is also written as Eq. (S15) in the link above.

We will use the notation $\delta x_{ij} = \delta x_i - \delta x_j$, $\delta y_{ij} = \delta y_i - \delta y_j$ and $r_{ij}^2 = (x_i - x_j)^2 + (y_i - y_j)^2$.

After setting $x_i \mapsto x_i + \delta x_i$ and $y_i \mapsto y_i + \delta y_i$ we arrive at the following expression for $(x_i - x_j) / |r_i - r_j|^3$

In [ ]:
Fx = (rij * cos(θij) + dxij) / (rij ** 3 * (cos(θij) ** 2 * (1 + dxij/(rij * cos(θij))) ** 2 + sin(θij) ** 2 * (1 + dyij/(rij * sin(θij))) ** 2 ) ** (Rational(3,2)))
Fx

To linearize we use the Taylor expansion around the equilibrium coordinates ($\delta x_i = 0, \delta y_i =0$)
$$ 
F_x \approx F_x (\delta x_i = 0, \delta y_i =0) + \frac{\partial F_x}{\partial \delta x_i} (\delta x_i = 0, \delta y_i =0) \delta x_i + \frac{\partial F_x}{\partial \delta y_i} (\delta x_i = 0, \delta y_i =0) \delta y_i
$$

The first term is simply setting $\delta x_i = 0, \delta y_i =0$, which is done with `subs`:

In [ ]:
first_term = Fx.subs(dxij, 0).subs(dyij, 0).simplify()
first_term 

However, note that the first term won't contribute to the EOM for $\delta x_i$ or $\delta y_i$. For the last two terms we need to differentiate $F_x$ w.r.t. $\delta x_i$ and $\delta y_i$, respectively

In [ ]:
linear_term_x = diff(Fx, dxij).subs(dxij, 0).subs(dyij, 0).simplify()
linear_term_x = linear_term_x.subs(sin(θij)**2, Rational(1, 2) - Rational(1 ,2) * cos(2*θij))
linear_term_x

And differentiating with respect to $\delta y_{ij}$ gives

In [ ]:
linear_term_y = diff(Fx, dyij).subs(dxij, 0).subs(dyij, 0).simplify()
linear_term_y

To summarize, we've now obtained the electron-electron interaction term $(x_i-x_j) / |r_i - r_j|^3$, which is given by the expression below:

In [ ]:
( first_term + linear_term_x * dxij + linear_term_y * dyij ).factor(dxij)

The first term won't contribute to the equations of motion, but the last two terms proportional to $\delta x_{ij}$ and $\delta y_{ij}$ will.

We can start to recognize $k_{ij}^+$ as the term proportional to $\delta x_{ij}$ and similarly $l_{ij}$ as the term proportional to $\delta y_{ij}$.
Note that we omitted $e^2 / 4\pi \varepsilon_0$ and a factor of $1/2$ from double counting. Taking these prefactors into account will give an exact comparison.

# Linearization of $F_y$ for the EOM in $\delta y_i$

The equation of motion for the coordinate $y_i$ contains the term $(y_i - y_j) / |r_i - r_j|^3$. We'll follow a similar procedure as above

In [ ]:
Fy = (rij * sin(θij) + dyij) / (rij ** 3 * (cos(θij) ** 2 * (1 + dxij/(rij * cos(θij))) ** 2 + sin(θij) ** 2 * (1 + dyij/(rij * sin(θij))) ** 2 ) ** (Rational(3, 2)))
Fy

To linearize this term we write 
$$ 
F_y \approx F_y (\delta x_i = 0, \delta y_i =0) + \frac{\partial F_y}{\partial \delta x_i} (\delta x_i = 0, \delta y_i =0) \delta x_i + \frac{\partial F_y}{\partial \delta y_i} (\delta x_i = 0, \delta y_i =0) \delta y_i
$$

Again, we start with the term that won't contribute to the equations of motion.

In [ ]:
first_term = Fy.subs(dxij, 0).subs(dyij, 0).simplify()
first_term 

Here is the term proportional to $\delta x_{ij}$

In [ ]:
linear_term_x = diff(Fy, dxij).subs(dxij, 0).subs(dyij, 0).simplify()
linear_term_x

and proportional to $\delta y_{ij}$

In [ ]:
linear_term_y = diff(Fy, dyij).subs(dxij, 0).subs(dyij, 0).simplify()
linear_term_y = linear_term_y.subs(sin(θij)**2, Rational(1, 2) - Rational(1, 2) * cos(2*θij))
linear_term_y

To summarize: the electron-electron interaction term for the $y$ EOM is given by $(y_i - y_j) / |r_i - r_j|^3$ and the linearized result is

In [ ]:
first_term + linear_term_x * dxij + linear_term_y * dyij

We can start to recognize $k_{ij}^-$ as the term proportional to $\delta y_{ij}$ and similarly $l_{ij}$ as the term proportional to $\delta x_{ij}$.
Note that we omitted $e^2 / 4\pi \varepsilon_0$ and a factor of $1/2$ from double counting. Taking these prefactors into account will give an exact comparison.

# Including the effects of screening

If we include screening, the regular Coulomb potential can be modified to a Yukawa potential

$$
U = \frac{e^{-r/\lambda}}{r}
$$
where $\lambda$ is the interaction cutoff length. We may repeat the previous sections to obtain the versions of $k_{ij}^\pm$ and $l_{ij}$ that include screening. These should be compared to Eqs. C.3 and C.4 in https://schusterlab.stanford.edu/static/pdfs/Koolstra_thesis.pdf Let's get started by figuring out the forces in the $x$ and $y$ direction.

## Force in the $x$ direction

Finding the force requires differentiating the potential. Because $r$ appears in the exponent and also in the denominator, it is not as straightforward. By hand, we would apply the chain rule. `sympy` doesn't care and will do it for you:

In [ ]:
rxy = sqrt(xij**2 + yij**2)
lamb = sp.symbols('λ')
xi = sp.symbols('ξ')

U = 1 / rxy * exp(-rxy / lamb)
Fx = diff(U, xij).simplify().subs(sqrt(xij**2 + yij**2), rij)
Fx

In [ ]:
linearized_rij = rij * sqrt(cos(θij) ** 2 * (1 + dxij/(rij * cos(θij))) ** 2 + sin(θij) ** 2 * (1 + dyij/(rij * sin(θij))) ** 2 )
linearized_Fx = Fx.subs(xij, xij + dxij).subs(rij, linearized_rij)
linearized_Fx

The resulting expressions start to grow, but let's not fear yet. Things will look better. 

In [ ]:
linearized_Fx = linearized_Fx.subs(xij, rij * cos(θij)).subs(yij, rij * sin(θij)).simplify()
linearized_Fx

Things will start to look more compact by using the substitution $\xi = r_{ij} / \lambda$. The first term in the Taylor expansion of the force is:

In [ ]:
first_term = linearized_Fx.subs(dxij, 0).subs(dyij, 0).simplify()
first_term.subs(lamb, rij / xi).simplify()

The term that is proportional to $\delta x_{ij}$ is found by differentiating w.r.t. $\delta x_{ij}$:

In [ ]:
linear_term_x = diff(linearized_Fx, dxij).subs(dxij, 0).subs(dyij, 0).simplify()
linear_term_x = linear_term_x.subs(lamb, rij / xi).simplify().subs(cos(θij)**2, Rational(1, 2) + Rational(1, 2) * cos(2*θij))
linear_term_x

The term that is proportional to $\delta y_{ij}$ is found by differentiating w.r.t. $\delta y_{ij}$:

In [ ]:
linear_term_y = diff(linearized_Fx, dyij).subs(dxij, 0).subs(dyij, 0).simplify()
linear_term_y = linear_term_y.subs(lamb, rij / xi).simplify().subs(cos(θij)**2, Rational(1, 2) + Rational(1, 2) * cos(2*θij))
linear_term_y

The final result below should be checked with $k_{ij}^+$ and $l_{ij}$ in C.3 and C.4 in https://schusterlab.stanford.edu/static/pdfs/Koolstra_thesis.pdf You can actually see there is a small error in C4!

In [ ]:
(linear_term_x * dxij + linear_term_y * dyij).factor(dxij, dyij)

## Force in the $y$-direction

You know the drill by know. First differentiate the potential energy (electron-electron interaction) w.r.t. $y_{ij}$ to find $F_y$ that corresponds to the Yukawa potential:

In [ ]:
Fy = diff(U, yij).simplify().subs(sqrt(xij**2 + yij**2), rij)
Fy

Perturb:

In [ ]:
linearized_Fy = Fy.subs(yij, yij + dyij).subs(rij, linearized_rij)
linearized_Fy

Group and simplify

In [ ]:
linearized_Fy = linearized_Fy.subs(xij, rij * cos(θij)).subs(yij, rij * sin(θij)).simplify()
linearized_Fy

Find the first term in the Taylor series that doesn't affect the EOM:

In [ ]:
first_term = linearized_Fy.subs(dxij, 0).subs(dyij, 0).simplify()
first_term.subs(lamb, rij / xi).simplify()

Find the term propoertional to $\delta x_{ij}$:

In [ ]:
linear_term_x = diff(linearized_Fy, dxij).subs(dxij, 0).subs(dyij, 0).simplify()
linear_term_x = linear_term_x.subs(lamb, rij / xi).simplify().subs(cos(θij)**2, Rational(1, 2) + Rational(1, 2) * cos(2*θij))
linear_term_x

Find the term propoertional to $\delta y_{ij}$ (it is actually rather compact given what we started off with)

In [ ]:
linear_term_y = diff(linearized_Fy, dyij).subs(dxij, 0).subs(dyij, 0).simplify()
linear_term_y = linear_term_y.subs(lamb, rij / xi).simplify().subs(sin(θij)**2, Rational(1, 2) - Rational(1, 2) * cos(2*θij))
linear_term_y

The final result below should be checked with $k_{ij}^+$ and $l_{ij}$ in C.3 and C.4 in https://schusterlab.stanford.edu/static/pdfs/Koolstra_thesis.pdf You can actually see there is a small error in C4!

In [ ]:
(linear_term_x * dxij + linear_term_y * dyij).factor(dxij, dyij)